In [1]:
device = 'cuda:4'
image_num = 1


In [2]:
from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch

local_path = "/disk1/users/fengyy/projects/models/llava-1.5-7b-hf"

# 指定单一 GPU，如 cuda:0
device = torch.device('cuda:1')

# 去掉 device_map，整个模型加载到指定 GPU
llava_model = LlavaForConditionalGeneration.from_pretrained(
    local_path,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    local_files_only=True
).to(device)  # 手动移动到指定 GPU

llava_processor = AutoProcessor.from_pretrained(local_path, local_files_only=True)
print("✓ 模型加载成功")

/disk1/users/fengyy/.conda/envs/Vattack/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 16.76it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ 模型加载成功


Dir

In [5]:
import os
import json
import torch
from glob import glob
from PIL import Image
from datetime import datetime

# Initialize conversation template for the description
description_conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the image briefly in one sentence."},
            {"type": "image"},
        ],
    },
]

# Set paths and parameters
adv_image_dir = '/disk1/users/fengyy/projects/V-Attack/adv_images/co/co_targeted_v/0_1_no_vision'
vqa_questions_path = 'coco300_vqa_main.json'
output_json = "co/llava_1.json"

# Load VQA questions
with open(vqa_questions_path, 'r') as f:
    vqa_data = json.load(f)

questions_map = {item['image']: item['vqa'] for item in vqa_data}

def create_question_conversation(question):
    return [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image"},
            ],
        },
    ]

# Load adversarial images
adv_files = []
for ext in ['*.jpg', '*.png', '*.jpeg']:
    adv_files.extend(glob(os.path.join(adv_image_dir, ext)))
adv_files.sort()

results = []
total = len(adv_files)

print(f"{'='*80}")
print(f"开始处理 {total} 张图片")
print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

with torch.no_grad():
    for i, adv_file in enumerate(adv_files, 1):
        filename = os.path.basename(adv_file).replace(".png", ".jpg")
        
        # 进度条
        progress = f"[{i}/{total}]"
        print(f"{progress} {'='*60}")
        print(f"处理文件: {filename}")
        
        if filename not in questions_map:
            print(f"  ⚠️ 跳过 - 未找到问题")
            continue
            
        questions = questions_map[filename]
        print(f"  问题数: {len(questions)}")
        
        # Load image
        adv_image = Image.open(adv_file).convert('RGB')
        
        # === 生成描述 ===
        print(f"\n  📷 [描述]")
        text = llava_processor.apply_chat_template(description_conversation, add_generation_prompt=True)
        inputs = llava_processor(images=adv_image, text=text, return_tensors='pt').to(device, torch.float16)
        output = llava_model.generate(**inputs, max_new_tokens=200, do_sample=False)
        begin_token = inputs['input_ids'].shape[1]
        description = llava_processor.decode(output[0][begin_token:], skip_special_tokens=True)
        print(f"     输出: {description[:100]}{'...' if len(description) > 100 else ''}")
        
        # === 回答问题 ===
        question_responses = []
        for q_idx, question in enumerate(questions[:3], 1):
            if not question.strip():
                print(f"\n  ❓ [问题{q_idx}] (空)")
                question_responses.append("")
                continue
            
            print(f"\n  ❓ [问题{q_idx}] {question[:60]}{'...' if len(question) > 60 else ''}")
            
            q_conversation = create_question_conversation(question)
            q_text = llava_processor.apply_chat_template(q_conversation, add_generation_prompt=True)
            q_inputs = llava_processor(images=adv_image, text=q_text, return_tensors='pt').to(device, torch.float16)
            q_output = llava_model.generate(**q_inputs, max_new_tokens=200, do_sample=False)
            q_begin_token = q_inputs['input_ids'].shape[1]
            q_response = llava_processor.decode(q_output[0][q_begin_token:], skip_special_tokens=True)
            question_responses.append(q_response)
            
            print(f"     回答: {q_response[:100]}{'...' if len(q_response) > 100 else ''}")
        
        # 保存结果
        result = {
            "filename": filename,
            "adversarial_response_1": description,
            "adversarial_response_2": question_responses[0],
            "adversarial_response_3": question_responses[1],
            "adversarial_response_4": question_responses[2],
        }
        results.append(result)
        
        # 实时显示完整结果
        print(f"\n  💾 已保存:")
        print(f"     描述: {description[:80]}...")
        for j, resp in enumerate(question_responses, 2):
            print(f"     回答{j-1}: {resp[:80]}{'...' if len(resp) > 80 else ''}")
        
        print(f"\n{'='*80}\n")
        
        # 每10张保存一次中间结果（防止崩溃丢失）
        if i % 10 == 0:
            with open(output_json, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=4)
            print(f"  💾 中间保存 ({i}/{total} 完成)\n")

# 最终保存
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\n{'='*80}")
print(f"✅ 全部完成！共处理 {len(results)}/{total} 张图片")
print(f"结果保存至: {output_json}")
print(f"结束时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}")

开始处理 1 张图片
时间: 2026-03-10 20:20:31

[1/1] ============================================================
处理文件: 000000000285.jpg
  问题数: 3

  📷 [描述]
     输出: A lion is laying down in a grassy field.

  ❓ [问题1] What is the large creature that occupies most of the frame?
     回答: The large creature occupying most of the frame is a lion.

  ❓ [问题2] Which animal in the image seems to be calmly observing its s...
     回答: The animal in the image that appears to be calmly observing its surroundings on the grassy area is a...

  ❓ [问题3] What kind of animal, known for its strength and size, is pro...
     回答: The image prominently features a lion, which is known for its strength and size. The lion is lying d...

  💾 已保存:
     描述: A lion is laying down in a grassy field....
     回答1: The large creature occupying most of the frame is a lion.
     回答2: The animal in the image that appears to be calmly observing its surroundings on ...
     回答3: The image prominently features a lion, which is known fo

FileNotFoundError: [Errno 2] No such file or directory: 'co/llava_1.json'

Dirs

In [ ]:
import os
import json
import torch
from glob import glob
from PIL import Image

# Initialize conversation template for the description
description_conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the image briefly in one sentence."},
            {"type": "image"},
        ],
    },
]

# Set paths and parameters
image_dirs = [
    '...',
    # Add more directories as needed
]
vqa_questions_path = 'coco300_vqa_main.json'
output_json_dir = "llava"  # Directory to store individual result files

# Load VQA questions
with open(vqa_questions_path, 'r') as f:
    vqa_data = json.load(f)

# Create a mapping from filename to questions for quick lookup
questions_map = {item['image']: item['vqa'] for item in vqa_data}

# Function to create conversation for a specific question
def create_question_conversation(question):
    return [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image"},
            ],
        },
    ]

# Store results in a list of dictionaries
results = []

# Function to process each file
def process_image_files(adv_image_dir):
    adv_files = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        adv_files.extend(glob(os.path.join(adv_image_dir, ext)))
    adv_files.sort()

    result_for_dir = []

    with torch.no_grad():
        for i, adv_file in enumerate(adv_files):
            filename = os.path.basename(adv_file).replace(".png", ".jpg")
            print(f'Processing: {filename}')
            
            # Skip if we don't have questions for this image
            if filename not in questions_map:
                print(f"Skipping {filename} - no questions found")
                continue
            
            # Get the questions for this image
            questions = questions_map[filename]
            if len(questions) < 3:
                print(f"Warning: Only {len(questions)} questions found for {filename}")
                questions.extend([""] * (3 - len(questions)))  # Pad with empty questions if needed
            
            # Load the adversarial image
            adv_image = Image.open(adv_file).convert('RGB')
            
            # Process description
            text = llava_processor.apply_chat_template(description_conversation, add_generation_prompt=True)
            inputs = llava_processor(images=adv_image, text=text, return_tensors='pt').to(device, torch.float16)
            output = llava_model.generate(**inputs, max_new_tokens=200, do_sample=False)
            begin_token = inputs['input_ids'].shape[1]
            description = llava_processor.decode(output[0][begin_token:], skip_special_tokens=True)
            
            # Process each question
            question_responses = []
            for question in questions[:3]:  # Only take first 3 questions
                if not question.strip():  # Skip empty questions
                    question_responses.append("")
                    continue
                    
                q_conversation = create_question_conversation(question)
                q_text = llava_processor.apply_chat_template(q_conversation, add_generation_prompt=True)
                q_inputs = llava_processor(images=adv_image, text=q_text, return_tensors='pt').to(device, torch.float16)
                q_output = llava_model.generate(**q_inputs, max_new_tokens=200, do_sample=False)
                q_begin_token = q_inputs['input_ids'].shape[1]
                q_response = llava_processor.decode(q_output[0][q_begin_token:], skip_special_tokens=True)
                question_responses.append(q_response)
            
            # Store results for this image
            result_for_dir.append({
                "filename": filename,
                "adversarial_response_1": description,
                "adversarial_response_2": question_responses[0],
                "adversarial_response_3": question_responses[1],
                "adversarial_response_4": question_responses[2],
            })
    
    # Save results for this directory
    if 'mattack' in adv_image_dir:
        output_json = os.path.join(output_json_dir, os.path.basename(adv_image_dir) + "_mattack_result.json")
    else:
        if 'ENS' in adv_image_dir:
            if 'targeted_v' in adv_image_dir:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_image_dir) + "_ens_v_result.json")
            else:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_image_dir) + "_ens_x_result.json")
            
        else:
            if 'targeted_v' in adv_image_dir:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_image_dir) + "_sgl_v_result.json")
            else:
                output_json = os.path.join(output_json_dir, os.path.basename(adv_image_dir) + "_sgl_x_result.json")

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(result_for_dir, f, ensure_ascii=False, indent=4)
    print(f"Results for {adv_image_dir} saved to {output_json}")

# Process each directory in the list
for image_dir in image_dirs:
    process_image_files(image_dir)
